# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Raja-saab/Flyrank1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule

I will prioritize content pages for refresh when they are both **stale** and have meaningful search visibility.

A page is considered:
- **stale** when it has not been updated for at least 180 days
- **visible** when it has at least 500 impressions in the observed 90-day window

The score combines staleness and search visibility. Higher scores mean higher priority for review.

This is a decision-support baseline, not a claim that every selected page definitely needs a refresh.

### Reason code

The rule uses one main reason code:

- `stale_visible_page` — the page is both stale and visibly receiving search impressions.

Pages that do not meet both conditions receive:

- `monitor` — not enough evidence for the refresh rule.

### Action

- `refresh` → prioritize the page for refresh review
- `monitor` → do not prioritize it for the refresh queue

The rule uses only information available at the decision moment. It does not use future performance, `trend_direction`, `trend_pct`, or any other label-derived field.

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

# -------------------------------------------------------
# Load the starter dataset
# -------------------------------------------------------

# Find the repository automatically.
candidates = [
    Path("/content/Flyrank1"),
    Path.cwd(),
]

REPO_ROOT = None

for candidate in candidates:
    if (candidate / "data/raw/content_refresh_anonymized.csv").exists():
        REPO_ROOT = candidate
        break

# Fallback: search /content
if REPO_ROOT is None:
    matches = list(
        Path("/content").glob(
            "**/data/raw/content_refresh_anonymized.csv"
        )
    )

    if matches:
        REPO_ROOT = matches[0].parents[2]

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv. "
        "Make sure you are running this notebook from your FlyRank repo."
    )

DATA_PATH = REPO_ROOT / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Repository:", REPO_ROOT)
print("Rows:", len(df))
print("Columns:", len(df.columns))

# -------------------------------------------------------
# Check required columns
# -------------------------------------------------------

required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "trend_direction",
    "trend_pct",
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

# -------------------------------------------------------
# Basic type cleaning
# -------------------------------------------------------

numeric_columns = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "trend_pct",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df["trend_direction"] = (
    df["trend_direction"]
    .fillna("unknown")
    .astype(str)
)

# -------------------------------------------------------
# Build the baseline population
# -------------------------------------------------------

baseline_df = df[
    (df["impressions_90d"] > 0)
    & (df["days_since_last_update"] >= 0)
].copy()

baseline_df = (
    baseline_df
    .drop_duplicates("content_id")
    .reset_index(drop=True)
)

print("Usable baseline rows:", len(baseline_df))

# -------------------------------------------------------
# Signal 1: staleness
# -------------------------------------------------------

baseline_df["staleness_bucket"] = pd.cut(
    baseline_df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=[
        "0-30d",
        "31-90d",
        "91-180d",
        "181-365d",
        "365+d",
    ],
)

staleness_table = (
    baseline_df
    .groupby(
        "staleness_bucket",
        observed=False
    )
    .agg(
        n=("content_id", "size"),
        median_impressions=(
            "impressions_90d",
            "median"
        ),
        declining_rate=(
            "trend_direction",
            lambda s: (
                s.str.lower() == "down"
            ).mean()
        ),
    )
    .reset_index()
)

print("\nSIGNAL 1 — STALENESS")
print(staleness_table.to_string(index=False))

# -------------------------------------------------------
# Signal 2: search visibility
# -------------------------------------------------------

baseline_df["visibility_bucket"] = pd.cut(
    baseline_df["impressions_90d"],
    bins=[0, 100, 500, 3000, 30000, np.inf],
    labels=[
        "1-100",
        "101-500",
        "501-3k",
        "3k-30k",
        "30k+",
    ],
)

visibility_table = (
    baseline_df
    .groupby(
        "visibility_bucket",
        observed=False
    )
    .agg(
        n=("content_id", "size"),
        median_days_since_update=(
            "days_since_last_update",
            "median"
        ),
        declining_rate=(
            "trend_direction",
            lambda s: (
                s.str.lower() == "down"
            ).mean()
        ),
    )
    .reset_index()
)

print("\nSIGNAL 2 — SEARCH VISIBILITY")
print(visibility_table.to_string(index=False))

Repository: /content/Flyrank1
Rows: 30000
Columns: 44
Usable baseline rows: 30000

SIGNAL 1 — STALENESS
staleness_bucket     n  median_impressions  declining_rate
           0-30d 20480               470.0        0.511377
          31-90d   175               510.0        0.588571
         91-180d  9171              1692.0        0.611057
        181-365d   169                16.0        0.467456
           365+d     5                 2.0        0.600000

SIGNAL 2 — SEARCH VISIBILITY
visibility_bucket    n  median_days_since_update  declining_rate
            1-100 8006                      20.0        0.389208
          101-500 5279                      22.0        0.604281
           501-3k 8432                      22.0        0.620849
           3k-30k 7205                      22.0        0.586121
             30k+ 1078                      25.0        0.461967


## 2. Build the ranked queue (writes the CSV)


I use a transparent rule-based score rather than a fitted model.

The score combines:

1. **Staleness risk** — pages that have gone longer without an update receive a higher score.
2. **Search visibility** — pages with more impressions receive a higher score.

The score is:

`baseline_action_score = staleness_rank × visibility_rank`

A page receives the `stale_visible_page` reason code and `refresh` action when:

- `days_since_last_update >= 180`, and
- `impressions_90d >= 500`.

All other pages receive the `monitor` action.

No future-window metric or label-derived field is used to calculate the score.

In [6]:
# -------------------------------------------------------
# Build the baseline score
# -------------------------------------------------------

queue = baseline_df.copy()

# Percentile rank for staleness.
# More days since update = greater staleness risk.
queue["staleness_rank"] = (
    queue["days_since_last_update"]
    .rank(
        pct=True,
        method="average"
    )
)

# Percentile rank for visibility.
# log1p prevents very large impression counts from dominating.
queue["visibility_rank"] = (
    np.log1p(queue["impressions_90d"])
    .rank(
        pct=True,
        method="average"
    )
)

# Transparent baseline score.
queue["baseline_action_score"] = (
    queue["staleness_rank"]
    * queue["visibility_rank"]
)

# -------------------------------------------------------
# Rule thresholds
# -------------------------------------------------------

queue["is_stale"] = (
    queue["days_since_last_update"] >= 180
)

queue["is_visible"] = (
    queue["impressions_90d"] >= 500
)

queue["rule_match"] = (
    queue["is_stale"]
    & queue["is_visible"]
)

# -------------------------------------------------------
# Exactly one reason code
# -------------------------------------------------------

queue["reason_code"] = np.where(
    queue["rule_match"],
    "stale_visible_page",
    "monitor"
)

# -------------------------------------------------------
# Action label
# -------------------------------------------------------

queue["action"] = np.where(
    queue["rule_match"],
    "refresh",
    "monitor"
)

# -------------------------------------------------------
# Rank everything
# -------------------------------------------------------

queue = (
    queue
    .sort_values(
        [
            "baseline_action_score",
            "impressions_90d"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue["baseline_rank"] = (
    np.arange(len(queue)) + 1
)

# -------------------------------------------------------
# Display top 10
# -------------------------------------------------------

display_columns = [
    "baseline_rank",
    "content_id",
    "baseline_action_score",
    "reason_code",
    "action",
    "impressions_90d",
    "days_since_last_update",
]

print("\nTOP 10 BASELINE QUEUE")

print(
    queue[display_columns]
    .head(10)
    .to_string(index=False)
)

# -------------------------------------------------------
# Write required CSV
# -------------------------------------------------------

OUTPUT_DIR = (
    REPO_ROOT
    / "work"
    / "outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    OUTPUT_DIR
    / "baseline_action_score.csv"
)

output_columns = [
    "content_id",
    "client_id",
    "baseline_rank",
    "baseline_action_score",
    "reason_code",
    "action",
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr",
]

queue[output_columns].to_csv(
    output_path,
    index=False
)

print(
    f"\n✅ CSV written to: {output_path}"
)

print(
    f"Rows written: {len(queue):,}"
)



TOP 10 BASELINE QUEUE
 baseline_rank           content_id  baseline_action_score        reason_code  action  impressions_90d  days_since_last_update
             1 content_cf56e2e2e282               0.982686 stale_visible_page refresh            61678                     194
             2 content_a5dbb404bdc2               0.982477            monitor monitor            79035                     106
             3 content_7368877ea310               0.982288 stale_visible_page refresh            59472                     194
             4 content_47b8b12d581e               0.968073            monitor monitor            40305                     106
             5 content_1bfaa38ff26c               0.951809 stale_visible_page refresh            25715                     194
             6 content_69fad7e6c50c               0.951654            monitor monitor            28000                     106
             7 content_482aff19e9cc               0.948417            monitor monitor   

## 3. Top-20 review

The following table reviews the 20 highest-ranked pages.

For each page I record:

- **action** — what the baseline recommends
- **reason code** — why it received that action
- **confidence note** — why the ranking looks reasonable
- **what would make it wrong** — a skeptical condition that could invalidate the recommendation

These are decision-support recommendations, not guaranteed outcomes.

In [7]:
# -------------------------------------------------------
# Top-20 review
# -------------------------------------------------------

top20 = queue.head(20).copy()

review = []

for _, row in top20.iterrows():

    if row["action"] == "refresh":
        confidence_note = (
            f"High staleness ({row['days_since_last_update']:.0f} days) "
            f"and meaningful visibility "
            f"({row['impressions_90d']:,.0f} impressions)."
        )
    else:
        confidence_note = (
            "The page ranks highly on the combined score, "
            "but does not satisfy both refresh thresholds."
        )

    wrong_if = (
        "The recommendation could be wrong if the old content is "
        "intentionally unchanged, the observed impressions are temporary, "
        "or refreshing the page would not address the real cause of performance."
    )

    review.append({
        "rank": int(row["baseline_rank"]),
        "content_id": row["content_id"],
        "action": row["action"],
        "reason_code": row["reason_code"],
        "confidence_note": confidence_note,
        "what_would_make_it_wrong": wrong_if,
    })

top20_review = pd.DataFrame(review)

display(top20_review)

,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_cf56e2e2e282,refresh,stale_visible_page,High staleness (194 days) and meaningful visib...,The recommendation could be wrong if the old c...
1,2,content_a5dbb404bdc2,monitor,monitor,"The page ranks highly on the combined score, b...",The recommendation could be wrong if the old c...
2,3,content_7368877ea310,refresh,stale_visible_page,High staleness (194 days) and meaningful visib...,The recommendation could be wrong if the old c...
3,4,content_47b8b12d581e,monitor,monitor,"The page ranks highly on the combined score, b...",The recommendation could be wrong if the old c...
4,5,content_1bfaa38ff26c,refresh,stale_visible_page,High staleness (194 days) and meaningful visib...,The recommendation could be wrong if the old c...
5,6,content_69fad7e6c50c,monitor,monitor,"The page ranks highly on the combined score, b...",The recommendation could be wrong if the old c...
6,7,content_482aff19e9cc,monitor,monitor,"The page ranks highly on the combined score, b...",The recommendation could be wrong if the old c...
7,8,content_6ac3ab740bbf,monitor,monitor,"The page ranks highly on the combined score, b...",The recommendation could be wrong if the old c...
8,9,content_cb7e312f5d32,monitor,monitor,"The page ranks highly on the combined score, b...",The recommendation could be wrong if the old c...
9,10,content_ac1d924c6a70,monitor,monitor,"The page ranks highly on the combined score, b...",The recommendation could be wrong if the old c...


## 4. Weak picks + leakage check

### Weak picks

I will inspect the highest-ranked rows for cases where the rule may be over-prioritizing a page.

Examples of possible weaknesses include:

- high impressions but relatively low staleness,
- stale pages with only modest visibility,
- unusual or missing position data.

A high baseline score is not proof that a refresh will work.

### Leakage check

The score is constructed only from:

- `days_since_last_update`
- `impressions_90d`

I deliberately exclude:

- `trend_direction`
- `trend_pct`
- any future-window metric
- any existing product/refresh flag
- any label-derived feature

The outcome label can be used later for evaluation, but never to construct the ranking.

In [8]:
# -------------------------------------------------------
# Weak-pick inspection
# -------------------------------------------------------

weak_picks = queue.head(20).copy()

weak_picks["weak_pick_reason"] = np.select(
    [
        weak_picks["days_since_last_update"] < 180,
        weak_picks["impressions_90d"] < 500,
        weak_picks["avg_position"].isna(),
    ],
    [
        "Not actually stale",
        "Below visibility threshold",
        "Missing position data",
    ],
    default="No obvious threshold weakness"
)

print("WEAK-PICK CHECK")

display(
    weak_picks[
        [
            "baseline_rank",
            "content_id",
            "action",
            "reason_code",
            "impressions_90d",
            "days_since_last_update",
            "avg_position",
            "weak_pick_reason",
        ]
    ]
)

# -------------------------------------------------------
# Leakage guard
# -------------------------------------------------------

score_inputs = {
    "days_since_last_update",
    "impressions_90d",
}

forbidden_inputs = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "future_window",
    "product_flag",
    "refresh_flag",
}

print("\nSCORE INPUTS:")
print(sorted(score_inputs))

print("\nFORBIDDEN / LABEL-DERIVED INPUTS:")
print(sorted(forbidden_inputs))

assert score_inputs.isdisjoint(
    forbidden_inputs
)

print(
    "\n✅ Leakage check passed."
)

# -------------------------------------------------------
# Confirm the actual score was created without labels
# -------------------------------------------------------

score_formula_columns = [
    "staleness_rank",
    "visibility_rank",
]

assert all(
    col in queue.columns
    for col in score_formula_columns
)

print(
    "✅ Score uses only decision-time signals."
)

# -------------------------------------------------------
# Basic queue checks
# -------------------------------------------------------

assert len(queue) > 0
assert queue["baseline_rank"].is_unique
assert queue["baseline_rank"].min() == 1
assert queue["baseline_rank"].max() == len(queue)

assert set(
    queue["action"].unique()
).issubset(
    {"refresh", "monitor"}
)

assert set(
    queue["reason_code"].unique()
).issubset(
    {"stale_visible_page", "monitor"}
)

print(
    "✅ Queue structure checks passed."
)

WEAK-PICK CHECK


,baseline_rank,content_id,action,reason_code,impressions_90d,days_since_last_update,avg_position,weak_pick_reason
0,1,content_cf56e2e2e282,refresh,stale_visible_page,61678,194,19.7,No obvious threshold weakness
1,2,content_a5dbb404bdc2,monitor,monitor,79035,106,8.7,Not actually stale
2,3,content_7368877ea310,refresh,stale_visible_page,59472,194,24.8,No obvious threshold weakness
3,4,content_47b8b12d581e,monitor,monitor,40305,106,28.4,Not actually stale
4,5,content_1bfaa38ff26c,refresh,stale_visible_page,25715,194,22.2,No obvious threshold weakness
5,6,content_69fad7e6c50c,monitor,monitor,28000,106,4.7,Not actually stale
6,7,content_482aff19e9cc,monitor,monitor,26287,106,13.2,Not actually stale
7,8,content_6ac3ab740bbf,monitor,monitor,22462,106,4.6,Not actually stale
8,9,content_cb7e312f5d32,monitor,monitor,21272,151,12.6,Not actually stale
9,10,content_ac1d924c6a70,monitor,monitor,21853,106,31.7,Not actually stale



SCORE INPUTS:
['days_since_last_update', 'impressions_90d']

FORBIDDEN / LABEL-DERIVED INPUTS:
['future_window', 'is_declining_label', 'product_flag', 'refresh_flag', 'trend_direction', 'trend_pct']

✅ Leakage check passed.
✅ Score uses only decision-time signals.
✅ Queue structure checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.